# 🚀 Path 1: Signal-to-Class
**Model**: 1D CNN

### Configuration:
- **Dataset**: MIT-BIH Arrhythmia (10% Sample for Testing)
- **Epochs**: 2
- **Environment**: Production-Ready Torch Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from torch.utils.data import DataLoader, TensorDataset, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = '../../data/mitbih_train.csv'

def load_production_data(sample_frac=0.1):
    if not os.path.exists(DATA_PATH): return None
    df = pd.read_csv(DATA_PATH, header=None)
    return df.sample(frac=sample_frac, random_state=42)

df_sample = load_production_data(0.1)
print(f'📦 Loaded {len(df_sample)} rows (10% of dataset)')

In [ ]:
class SimpleCNN1D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 46, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

model = SimpleCNN1D().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Data Prep
X = torch.tensor(df_sample.iloc[:, :187].values, dtype=torch.float32).unsqueeze(1).to(DEVICE)
y = torch.tensor(df_sample.iloc[:, 187].values, dtype=torch.long).to(DEVICE)
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)

print('🚀 Production Training: Signal-to-Class')
for epoch in range(2):
    model.train()
    total_loss, correct = 0, 0
    for signals, labels in loader:
        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    acc = correct / len(df_sample)
    print(f'Epoch {epoch+1}: Loss={avg_loss:.4f}, Accuracy={acc:.4f}')

if not os.path.exists('models'): os.makedirs('models')
torch.save(model.state_dict(), 'models/signal_to_class.pth')
print('✅ Model Saved to models/signal_to_class.pth')